# OpenCV pose estimation test

Notebook ORB + matrice essentielle entre deux frames reelles du dataset.

In [1]:
import cv2
import numpy as np

In [2]:
frame1_path = r"C:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\dataset\real\frames_real\frame_00000.png"
frame2_path = r"C:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\dataset\real\frames_real\frame_00001.png"

def read_grayscale_unicode(path):
    buffer = np.fromfile(path, dtype=np.uint8)
    if buffer.size == 0:
        raise FileNotFoundError(f"Impossible de lire le fichier: {path}")
    image = cv2.imdecode(buffer, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"cv2.imdecode a echoue pour: {path}")
    return image

frame1 = read_grayscale_unicode(frame1_path)
frame2 = read_grayscale_unicode(frame2_path)

print("frame1 shape:", frame1.shape)
print("frame2 shape:", frame2.shape)

frame1 shape: (480, 640)
frame2 shape: (480, 640)


In [3]:
fx = 525.0
fy = 525.0
cx = 319.5
cy = 239.5

K = np.array([
    [fx,  0, cx],
    [ 0, fy, cy],
    [ 0,  0,  1]
], dtype=np.float64)

print(K)

[[525.    0.  319.5]
 [  0.  525.  239.5]
 [  0.    0.    1. ]]


In [4]:
orb = cv2.ORB_create(nfeatures=3000)

kp1, des1 = orb.detectAndCompute(frame1, None)
kp2, des2 = orb.detectAndCompute(frame2, None)

if des1 is None or des2 is None:
    raise RuntimeError("ORB n'a pas trouve assez de descripteurs sur une des deux images.")

print("keypoints frame1:", len(kp1))
print("keypoints frame2:", len(kp2))

keypoints frame1: 2956
keypoints frame2: 2974


In [5]:
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)

if len(matches) < 8:
    raise RuntimeError(f"Pas assez de matches pour estimer la pose: {len(matches)}")

print("matches:", len(matches))
print("best distance:", matches[0].distance)
print("worst distance:", matches[-1].distance)

matches: 2031
best distance: 1.0
worst distance: 72.0


In [6]:
pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

print("pts1 shape:", pts1.shape)
print("pts2 shape:", pts2.shape)

pts1 shape: (2031, 2)
pts2 shape: (2031, 2)


In [7]:
E, mask = cv2.findEssentialMat(
    pts1,
    pts2,
    K,
    method=cv2.RANSAC,
    prob=0.999,
    threshold=1.0
)

if E is None:
    raise RuntimeError("findEssentialMat a echoue.")

print("E shape:", E.shape)
print(E)

E shape: (3, 3)
[[-5.67220883e-03 -6.65359821e-01  1.67019764e-01]
 [ 6.65632937e-01 -3.61435723e-03  1.74805158e-01]
 [-1.63605215e-01 -1.70189959e-01  3.60205280e-04]]


In [8]:
_, R, t, mask_pose = cv2.recoverPose(E, pts1, pts2, K)

print("Rotation R:")
print(R)
print()
print("Translation direction t:")
print(t)

Rotation R:
[[ 0.99996056 -0.00722949  0.00515972]
 [ 0.00726548  0.99994917 -0.00699045]
 [-0.00510892  0.00702766  0.99996225]]

Translation direction t:
[[ 0.24235731]
 [-0.22962093]
 [-0.94262249]]


In [9]:
T = np.eye(4)
T[:3, :3] = R
T[:3, 3] = t.ravel()

print(T)

[[ 0.99996056 -0.00722949  0.00515972  0.24235731]
 [ 0.00726548  0.99994917 -0.00699045 -0.22962093]
 [-0.00510892  0.00702766  0.99996225 -0.94262249]
 [ 0.          0.          0.          1.        ]]


Resultat : `T` est la pose relative entre `frame1` et `frame2`.

Attention : avec une seule camera, `t` donne surtout la direction du deplacement, pas la vraie distance en metres.

-----

### Comparaison pose obentue et GT

frame 1 : 1305031098.6659 1.3563 0.6305 1.6380 0.6132 0.5962 -0.3311 -0.3986
frame 2 : 1305031098.6758 1.3543 0.6306 1.6360 0.6129 0.5966 -0.3316 -0.3980

In [10]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [11]:
import numpy as np
from scipy.spatial.transform import Rotation as Rot

# Frame 1
gt1 = [1305031098.6659, 1.3563, 0.6305, 1.6380,
       0.6132, 0.5962, -0.3311, -0.3986]

# Frame 2
gt2 = [1305031098.6758, 1.3543, 0.6306, 1.6360,
       0.6129, 0.5966, -0.3316, -0.3980]

def gt_to_T(gt):
    _, tx, ty, tz, qx, qy, qz, qw = gt

    T = np.eye(4)
    T[:3, :3] = Rot.from_quat([qx, qy, qz, qw]).as_matrix()
    T[:3, 3] = [tx, ty, tz]

    return T

# Poses absolues
T1 = gt_to_T(gt1)
T2 = gt_to_T(gt2)

# Mouvement de la caméra entre frame1 et frame2
T_motion = np.linalg.inv(T1) @ T2

print("Matrice de mouvement GT :")
print(T_motion)

Matrice de mouvement GT :
[[ 9.99998294e-01  5.25147690e-05 -1.84625022e-03 -1.78578996e-04]
 [-5.22094598e-05  9.99999985e-01  1.65415015e-04  8.35727846e-04]
 [ 1.84625888e-03 -1.65318341e-04  9.99998282e-01  2.69808608e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [12]:
print("T estimé")
print(T)
print("T motion")
print(T_motion)

T estimé
[[ 0.99996056 -0.00722949  0.00515972  0.24235731]
 [ 0.00726548  0.99994917 -0.00699045 -0.22962093]
 [-0.00510892  0.00702766  0.99996225 -0.94262249]
 [ 0.          0.          0.          1.        ]]
T motion
[[ 9.99998294e-01  5.25147690e-05 -1.84625022e-03 -1.78578996e-04]
 [-5.22094598e-05  9.99999985e-01  1.65415015e-04  8.35727846e-04]
 [ 1.84625888e-03 -1.65318341e-04  9.99998282e-01  2.69808608e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [13]:
print( T- T_motion)

[[-3.77391560e-05 -7.28200232e-03  7.00596596e-03  2.42535887e-01]
 [ 7.31768485e-03 -5.08129882e-05 -7.15586193e-03 -2.30456658e-01]
 [-6.95517501e-03  7.19297731e-03 -3.60272172e-05 -9.45320579e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]
